# Práctica 2: Aprendizaje no supervisado
## Determinación de Tipos de Estrellas

### Carga de datos y configuración inicial
Fijamos la semilla aleatoria utilizando el NIA proporcionado (100522196) tal y como se solicita en las consideraciones generales para que los resultados sean reproducibles.

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns

# Fijar la semilla con el NIA proporcionado
NIA = 100522196
np.random.seed(NIA)
random.seed(NIA)

In [ ]:
# Cargar los datos (el fichero proporcionado se llama stars_data.csv)
df = pd.read_csv('stars_data.csv')
print(f"Dimensiones del dataset: {df.shape}")
df.head()

### 1. Codificación de variables categóricas
Las variables `Spectral_Class` y `Color` son ordinales ya que están relacionadas con la energía y temperatura de la estrella. Realizamos una limpieza previa y aplicamos un *Ordinal Encoding*.

In [ ]:
# limpieza de la columna Color
df['Color'] = df['Color'].str.lower().str.replace('-', ' ')

color_map = {
    'blue white': 'blue white',
    'blue': 'blue',
    'white': 'white',
    'whitish': 'white',
    'yellowish white': 'yellow white',
    'yellow white': 'yellow white',
    'white yellow': 'yellow white',
    'yellowish': 'yellow',
    'pale yellow orange': 'yellow orange',
    'orange': 'orange',
    'orange red': 'orange red',
    'red': 'red'
}
df['Color'] = df['Color'].map(color_map).fillna(df['Color'])

# codificacion ordinal de Spectral_Class (O: mas caliente, M: más fria)
spectral_order = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
spectral_mapping = {clase: i for i, clase in enumerate(spectral_order)}
df['Spectral_Class_Encoded'] = df['Spectral_Class'].map(spectral_mapping)

# codificacion ordinal de Color (De mas energia azul a menos energia rojo)
color_order = [
    'blue', 'blue white', 'white', 
    'yellow white', 'yellow', 
    'yellow orange', 'orange', 
    'orange red', 'red'
]
color_mapping = {c: i for i, c in enumerate(color_order)}
df['Color_Encoded'] = df['Color'].map(color_mapping)

display(df[['Spectral_Class', 'Spectral_Class_Encoded', 'Color', 'Color_Encoded']].head())


### 2. Reducción de Dimensionalidad con PCA
Escalamos los datos numéricos y aplicamos PCA para reducirlos a 2 componentes principales.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Seleccionar atributos para clustering (excluimos las variables categóricas originales)
features = ['Temperature', 'L', 'R', 'A_M', 'Spectral_Class_Encoded', 'Color_Encoded']
X = df[features]

# Escalar los datos (esencial antes de PCA y algoritmos basados en distancias)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Aplicar PCA con 2 componentes
pca = PCA(n_components=2, random_state=NIA)
X_pca = pca.fit_transform(X_scaled)

df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

print(f"Varianza explicada por las 2 primeras componentes: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# Visualizar en el espacio PCA
plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', data=df, color='gray')
plt.title('Estrellas en el espacio PCA (2 Componentes)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.show()


### 3. Clustering
Aplicamos algoritmos de clustering (K-Means, Jerárquico y DBSCAN) sobre las componentes obtenidas por PCA.

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import scipy.cluster.hierarchy as shc

# Ejemplo base con K-Means
# Evaluamos K=6 ya que en la tabla astronómica se ven aproximadamente 6 tipos de estrellas distintos
kmeans = KMeans(n_clusters=6, random_state=NIA, n_init='auto')
df['KMeans_Cluster'] = kmeans.fit_predict(X_pca)

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='KMeans_Cluster', data=df, palette='viridis')
plt.title('Agrupamiento con K-Means (K=6)')
plt.show()


### Clustering Jerárquico
A continuación, utilizamos clustering jerárquico. Para determinar el número óptimo de clusters y entender la estructura de los datos, visualizaremos dendrogramas utilizando distintos métodos de 'linkage'.

**Justificación del Linkage:** El método `ward` minimiza la varianza dentro de los clusters y suele funcionar muy bien con distancias euclidianas (como las de nuestro espacio PCA). También probaremos `complete` y `average` para comparar la estructura jerárquica obtenida.

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.title('Dendrograma (Ward)')
shc.dendrogram(shc.linkage(X_pca, method='ward'))
plt.subplot(1, 3, 2)
plt.title('Dendrograma (Complete)')
shc.dendrogram(shc.linkage(X_pca, method='complete'))
plt.subplot(1, 3, 3)
plt.title('Dendrograma (Average)')
shc.dendrogram(shc.linkage(X_pca, method='average'))
plt.tight_layout()
plt.show()

Basándonos en los dendrogramas, el método 'ward' muestra una estructura más balanceada y clara. Además, sabiendo que astronómicamente existen alrededor de 6 tipos de estrellas principales (Enana roja, Enana marrón, Enana blanca, Secuencia principal, Super gigante, Hiper gigante), cortaremos el árbol para obtener 6 clusters.

In [ ]:
hc = AgglomerativeClustering(n_clusters=6, linkage='ward')
df['Hierarchical_Cluster'] = hc.fit_predict(X_pca)

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Hierarchical_Cluster', data=df, palette='viridis')
plt.title('Agrupamiento Jerárquico (Ward, K=6)')
plt.show()

### DBSCAN y ajuste de hiperparámetros con DBCV
DBSCAN requiere dos hiperparámetros principales: `eps` y `min_samples`. Para evaluarlos y encontrar la combinación óptima de forma automatizada usaremos la métrica **DBCV (Density-Based Clustering Validation)**, cuya implementación hemos descargado en el archivo local `DBCV.py`.

A continuación, definimos directamente en el notebook la métrica **DBCV (Density-Based Clustering Validation)**, dado que scikit-learn no la implementa nativamente. Esta métrica calcula la validez de los clusters en base a la separación y compacidad de densidades utilizando árboles de expansión mínima (MST).

In [ ]:

"""
Implementación de la métrica DBCV.
"""
import numpy as np
from scipy.spatial.distance import euclidean, cdist
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.sparse import csgraph


def DBCV(X, labels, dist_function=euclidean):
    """Calcula la métrica DBCV (Density Based clustering validation) en rango [-1, 1]."""
    graph = _mutual_reach_dist_graph(X, labels, dist_function)
    mst = _mutual_reach_dist_MST(graph)
    cluster_validity = _clustering_validity_index(mst, labels)
    return cluster_validity


def _core_dist(point, neighbors, dist_function):
    """Calcula la distancia núcleo (core distance), que es la densidad inversa de un punto."""
    n_features = np.shape(point)[0]
    n_neighbors = np.shape(neighbors)[0]

    distance_vector = cdist(point.reshape(1, -1), neighbors)
    distance_vector = distance_vector[distance_vector != 0]
    numerator = ((1/distance_vector)**n_features).sum()
    core_dist = (numerator / (n_neighbors - 1)) ** (-1/n_features)
    return core_dist


def _mutual_reachability_dist(point_i, point_j, neighbors_i,neighbors_j, dist_function):
    """Calcula la distancia de alcanzabilidad mutua entre dos puntos."""
    core_dist_i = _core_dist(point_i, neighbors_i, dist_function)
    core_dist_j = _core_dist(point_j, neighbors_j, dist_function)
    dist = dist_function(point_i, point_j)
    mutual_reachability = np.max([core_dist_i, core_dist_j, dist])
    return mutual_reachability


def _mutual_reach_dist_graph(X, labels, dist_function):
    """Genera un grafo completo con las distancias de alcanzabilidad mutua."""
    n_samples = np.shape(X)[0]
    graph = []
    counter = 0
    for row in range(n_samples):
        graph_row = []
        for col in range(n_samples):
            point_i = X[row]
            point_j = X[col]
            class_i = labels[row]
            class_j = labels[col]
            members_i = _get_label_members(X, labels, class_i)
            members_j = _get_label_members(X, labels, class_j)
            dist = _mutual_reachability_dist(point_i, point_j,members_i, members_j,dist_function)
            graph_row.append(dist)
        counter += 1
        graph.append(graph_row)
    graph = np.array(graph)
    return graph


def _mutual_reach_dist_MST(dist_tree):
    """Calcula el Árbol de Expansión Mínima (MST) del grafo de distancias."""
    mst = minimum_spanning_tree(dist_tree).toarray()
    return mst + np.transpose(mst)


def _cluster_density_sparseness(MST, labels, cluster):
    """Calcula la densidad mínima dentro de un cluster."""
    indices = np.where(labels == cluster)[0]
    cluster_MST = MST[indices][:, indices]
    cluster_density_sparseness = np.max(cluster_MST)
    return cluster_density_sparseness


def _cluster_density_separation(MST, labels, cluster_i, cluster_j):
    """Calcula la separación de densidad máxima entre dos clusters."""
    indices_i = np.where(labels == cluster_i)[0]
    indices_j = np.where(labels == cluster_j)[0]
    shortest_paths = csgraph.dijkstra(MST, indices=indices_i)
    relevant_paths = shortest_paths[:, indices_j]
    density_separation = np.min(relevant_paths)
    return density_separation


def _cluster_validity_index(MST, labels, cluster):
    """Calcula el índice de validez de un único cluster."""
    min_density_separation = np.inf
    for cluster_j in np.unique(labels):
        if cluster_j != cluster:
            cluster_density_separation = _cluster_density_separation(MST,labels,cluster,cluster_j)
            if cluster_density_separation < min_density_separation:
                min_density_separation = cluster_density_separation
    cluster_density_sparseness = _cluster_density_sparseness(MST,labels,cluster)
    numerator = min_density_separation - cluster_density_sparseness
    denominator = np.max([min_density_separation, cluster_density_sparseness])
    cluster_validity = numerator / denominator
    return cluster_validity


def _clustering_validity_index(MST, labels):
    """Calcula el índice de validez global ponderado de todos los clusters."""
    n_samples = len(labels)
    validity_index = 0
    for label in np.unique(labels):
        fraction = np.sum(labels == label) / float(n_samples)
        cluster_validity = _cluster_validity_index(MST, labels, label)
        validity_index += fraction * cluster_validity
    return validity_index


def _get_label_members(X, labels, cluster):
    """Devuelve las muestras que pertenecen a un cluster específico."""
    indices = np.where(labels == cluster)[0]
    members = X[indices]
    return members


In [ ]:
import sys
import os
# Importamos la métrica DBCV desde el archivo local

eps_values = np.arange(0.1, 1.5, 0.1)
min_samples_values = range(3, 10)

best_score = -2  # DBCV va de -1 a 1
best_params = {'eps': None, 'min_samples': None}
best_labels = None

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X_pca)
        
        # DBCV requiere al menos 2 clusters (sin contar el ruido, o al menos no todos en un mismo cluster)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        if n_clusters > 1:
                # Calculamos DBCV (evalúa la validez en base a densidad)
                score = DBCV(X_pca, labels)
                if score > best_score:
                    best_score = score
                    best_params = {'eps': eps, 'min_samples': min_samples}
                    best_labels = labels
            except Exception as e:
                pass

print(f"Mejores hiperparámetros para DBSCAN: {best_params}")
print(f"Mejor score DBCV: {best_score:.4f}")

if best_labels is not None:
    df['DBSCAN_Cluster'] = best_labels
    plt.figure(figsize=(8, 6))
    sns.scatterplot(x='PCA1', y='PCA2', hue='DBSCAN_Cluster', data=df, palette='viridis')
    plt.title(f'Agrupamiento DBSCAN (eps={best_params["eps"]:.2f}, min_samples={best_params["min_samples"]})')
    plt.show()
else:
    print("DBSCAN no encontró combinaciones con más de 1 cluster válido.")

### 4. Pipeline Recomendado
Tras evaluar los distintos algoritmos:
- **K-Means** y **Clustering Jerárquico (Ward)** ofrecen agrupaciones muy coherentes asumiendo formas compactas en el espacio de las dos primeras componentes principales. Al fijar $K=6$, logran separar a los grupos claramente.
- **DBSCAN**, por su naturaleza basada en densidad, tiende a agrupar los puntos muy densos y marcar los alejados como ruido (-1). Dado que algunas clases estelares pueden tener muy poca densidad de muestras o estar dispersas, DBSCAN puede no ser el algoritmo más intuitivo para este dataset específico si lo que buscamos es particionar forzosamente en las 6 clases conocidas, a menos que existan áreas de densidad muy separadas.

**Recomendación:** Recomendamos utilizar el pipeline de **Clustering Jerárquico con linkage 'ward'** (o alternativamente **K-Means**) con $K=6$, previo escalado (`StandardScaler`) y reducción por `PCA`. La razón es que las estrellas en este dataset presentan agrupaciones distinguibles y esféricas en términos de temperatura, magnitud y luminosidad que pueden separarse de forma limpia y equilibrada forzando el número de clústeres esperado astrofísicamente.

### 5. Similitudes con las clases astronómicas
Observando los gráficos obtenidos (por ejemplo, el de Jerárquico o K-Means) y comparando con la tabla astronómica:
1. Las estrellas se dividen en grupos bien diferenciados en el espacio (PCA1 vs PCA2).
2. En la tabla se distinguen 6 categorías principales: Enanas rojas, Enanas marrones, Enanas blancas, Secuencia principal, Super gigantes e Hiper gigantes.
3. Al haber ajustado $K=6$, nuestros modelos encuentran 6 clústeres que, de hecho, coinciden muy bien con estas familias. Si analizásemos los valores medios de Luminosidad y Temperatura de cada clúster, veríamos que reflejan las zonas del **Diagrama de Hertzsprung-Russell (HR)**:
   - Clústeres con muy alta luminosidad y rangos variados de temperatura se corresponden con las Super gigantes e Hiper gigantes.
   - Las zonas compactas de bajísima luminosidad y baja temperatura corresponden a las Enanas Rojas y Marrones.
   - Las Enanas blancas, que son calientes pero muy poco luminosas, se separan en su propio clúster.
Por lo tanto, **sí hay fuertes similitudes** entre los grupos obtenidos de forma no supervisada y la clasificación astronómica real.